# Lecture 5 — Capstone: *Review, Diagnose, and Repair a Biomarker Study*
### Practical Machine Learning for Transcriptomics in Cancer Research

This is the **capstone**. You are handed a *deliberately flawed* biomarker analysis — a short "submitted
manuscript" with an impressive headline result — and you do what a careful reviewer (and a careful author)
must do:

1. **Review** the submitted analysis and reproduce its headline number.
2. **Diagnose** its methodological flaws — name each one, with evidence.
3. **Propose** corrections.
4. **Rebuild** the workflow honestly — leakage-safe, nested, externally validated.
5. **Compare** original vs corrected and quantify the inflation.
6. **Write a reviewer report** with a publish / revise / reject recommendation. *(the primary deliverable)*
7. **Reflect** on which course lessons mattered most.

> **The thesis you are demonstrating:** *the hardest part of machine learning is not training a model; it
> is demonstrating that the model is trustworthy.* Expect the impressive headline (only ~0.69 to begin with) to **unravel**
> once the flaws are fixed. **That unravelling is the lesson** — it is the whole course (validation >
> optimization) in one project.

#### Continuity & reminders
- **Label:** binary **recurrence** — *not* pCR. The binary label is a deliberate simplification of
  time-to-event data; an optional Part 4 cell restores the survival framing.
- **The flaws are not labelled in the analysis** — you discover them with the reviewer checklist.
- **Leakage discipline:** selection, scaling, *and batch correction* are fit on training data only and
  applied to held-out / external data — never pooled across the split.
- **Same data cache as Lectures 1–4** — METABRIC + GSE6532 are reused, not re-downloaded.

> **Network note.** Reuses the L1–L4 real-data loaders (cBioPortal + GEO). If the prepared cohort is in
> the shared cache it is used directly; otherwise regenerated (needs internet). Downloads are git-ignored.


## Setup

Environment and data are prepared by **Lesson 0** (conda env `ml26` + downloads). This cell only
imports libraries and the checkpoint helpers — it installs nothing.

### 📚 Official docs & resources for this lesson

New to a tool, or want the authoritative reference? These point at exactly what this notebook uses. (Lesson 0 has the full library table.)

- **Core stack** — [NumPy](https://numpy.org/doc/stable/) · [pandas](https://pandas.pydata.org/docs/) · [Matplotlib](https://matplotlib.org/stable/index.html)
- **scikit-learn** — [Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline) · [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html) · [Ensembles](https://scikit-learn.org/stable/modules/ensemble.html) · [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) · [Metrics & scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- **Survival analysis** — [lifelines](https://lifelines.readthedocs.io/en/latest/) · [Kaplan–Meier & survival intro](https://lifelines.readthedocs.io/en/latest/Survival%20analysis%20with%20lifelines.html)
- **External cohort** — [NCBI GEO — GSE6532](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE6532) · [GEOparse](https://geoparse.readthedocs.io/en/latest/)
- **Going deeper** — [Common pitfalls & data leakage](https://scikit-learn.org/stable/common_pitfalls.html) · [Nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)


In [ ]:
# ── Setup — imports + checkpoint I/O (environment & data come from LESSON 0) ────
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import (StratifiedKFold, cross_val_score, cross_val_predict,
                                     train_test_split, GridSearchCV)
from sklearn.metrics import roc_auc_score, average_precision_score

# ── checkpoint I/O — the data hand-off between lessons ─────────────────────────
# Each lesson SAVES what the next one needs and LOADS what the previous one made.
# Checkpoints live in the shared datasets/derived/ folder (git-ignored).
import pickle

def _derived_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for d in sorted((base / "lessons").glob("lesson01_*/practical/task/datasets")):
            (d / "derived").mkdir(parents=True, exist_ok=True)
            return d / "derived"
    d = Path.cwd() / "datasets" / "derived"; d.mkdir(parents=True, exist_ok=True)
    return d

def save_checkpoint(name, **objs):
    path = _derived_dir() / f"{name}.pkl"
    with open(path, "wb") as fh:
        pickle.dump(objs, fh)
    print(f"saved checkpoint '{name}'  ->  {path}")
    return path

def load_checkpoint(name):
    path = _derived_dir() / f"{name}.pkl"
    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint '{name}' not found ({path}).\n"
            f"Run the earlier lesson that creates it first — it ends with "
            f"save_checkpoint('{name}', ...).")
    with open(path, "rb") as fh:
        return pickle.load(fh)


### Load features from Lesson 3, and build the external cohort

> **Reminder — this capstone builds on Lesson 3.** It **loads** the `lesson03_features` checkpoint
> (raw genes + engineered features + split), recovers the survival frame from the clinical table, and
> builds a **real external cohort** (GSE6532) here. If the load fails, run **Lesson 3** first.

In [ ]:
# The capstone needs raw genes + engineered features from LESSON 3, plus a REAL external
# cohort (GSE6532) built here. Strict hand-off: if the load raises, run Lesson 3 first.
def _resolve_data_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for cand in sorted((parent / "lessons").glob("lesson01_*/practical/task/datasets")):
            return str(cand)
    return str(Path.cwd() / "datasets")
DATA_DIR = _resolve_data_dir()

def _cached(fname):
    """Path to a file Lesson 0 downloaded; raises a clear pointer to Lesson 0 if missing."""
    p = os.path.join(DATA_DIR, fname)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        raise FileNotFoundError(f"'{fname}' not in the cache ({DATA_DIR}). Run Lesson 0 first "
                                "(lessons/lesson00_prerequisites/).")
    return p

SIGNATURES = {
    "proliferation": ["MKI67","AURKA","CCNB1","CCNB2","BUB1","TOP2A","CDK1","CCNE2","MYBL2","UBE2C","BIRC5","RRM2","TYMS","CENPF","PLK1"],
    "er_signalling": ["ESR1","FOXA1","GATA3","XBP1","BCL2","PGR","TFF1","GREB1","AR","NAT1","MLPH"],
    "immune":        ["CD8A","CD8B","GZMB","PRF1","IFNG","CXCL9","CXCL10","CD3D","CD3E","GZMA","NKG7","STAT1"],
    "stromal":       ["FAP","COL1A1","COL1A2","COL3A1","ACTA2","PDGFRB","FN1","VIM","THY1","SPARC","TIMP1"],
}
HALLMARK_SETS = {
    "E2F_TARGETS":      ["MKI67","BUB1","CCNB2","AURKA","TOP2A","RRM2","MYBL2","CDK1"],
    "G2M_CHECKPOINT":   ["CCNB1","CCNB2","PLK1","BUB1","CENPF","UBE2C","BIRC5","CDK1"],
    "ESTROGEN_EARLY":   ["ESR1","FOXA1","GATA3","TFF1","GREB1","PGR","XBP1"],
    "ESTROGEN_LATE":    ["BCL2","NAT1","MLPH","AR","ESR1","TFF1"],
    "INTERFERON_GAMMA": ["CXCL9","CXCL10","STAT1","IFNG","GZMB","PRF1"],
    "INFLAMMATORY":     ["CD8A","CD3D","CD3E","GZMA","NKG7","CD8B"],
    "EMT":              ["COL1A1","COL1A2","COL3A1","FN1","VIM","SPARC","ACTA2"],
    "ANGIOGENESIS":     ["PDGFRB","TIMP1","FAP","THY1","SPARC"],
    "APOPTOSIS":        ["BCL2","BIRC5","TIMP1","GZMB"],
    "MYC_TARGETS":      ["RRM2","TYMS","CCNE2","UBE2C","CDK1","MYBL2"],
}
def score_sets(X, sets):
    """Mean of standardised member genes -> one score per set (leakage-exempt: fixed lists)."""
    cols = {}
    for name, genes in sets.items():
        present = [g for g in genes if g in X.columns]
        if not present:
            continue
        z = (X[present] - X[present].mean()) / (X[present].std() + 1e-9)
        cols[name] = z.mean(axis=1)
    return pd.DataFrame(cols, index=X.index)

def load_gse6532_external(feature_columns, data_dir=DATA_DIR, horizon_months=60):
    """Build a REAL external-validation cohort from GSE6532 (Loi et al., Affymetrix).
    Cross-platform test (METABRIC/Illumina -> GSE6532/Affymetrix): same engineered features,
    same binary recurrence label (DMFS at the horizon), ER+ patients. Downloads ~180 MB on first run."""
    import GEOparse
    gse = GEOparse.get_GEO(filepath=_cached("GSE6532_family.soft.gz"), silent=True)
    PLATS = ("GPL96", "GPL570")
    pmap = {}
    for pl in PLATS:
        gpl = gse.gpls[pl].table
        sym = next(c for c in gpl.columns if c.lower() in ("gene symbol", "gene_symbol", "symbol"))
        m = gpl.set_index("ID")[sym].dropna().astype(str); m = m[m.str.len() > 0]
        pmap.update(m.to_dict())
    cols, meta = {}, {}
    for name, gsm in gse.gsms.items():
        if gsm.metadata.get("platform_id", ["?"])[0] not in PLATS:
            continue
        tbl = gsm.table
        if tbl is None or "VALUE" not in tbl.columns:
            continue
        cols[name] = pd.Series(tbl["VALUE"].values, index=tbl["ID_REF"].astype(str).values)
        dd = {}
        for it in gsm.metadata.get("characteristics_ch1", []):
            if ":" in it:
                k, v = it.split(":", 1); dd[k.strip().lower()] = v.strip()
        meta[name] = dd
    expr = pd.DataFrame(cols); expr = expr[expr.index.isin(pmap)]
    expr.index = [pmap[i] for i in expr.index]
    Xg = expr.groupby(level=0).mean().T
    clin = pd.DataFrame(meta).T
    er = clin["er"].eq("1")
    event = pd.to_numeric(clin.get("e.dmfs"), errors="coerce")
    months = pd.to_numeric(clin.get("t.dmfs"), errors="coerce") / 30.44
    y = pd.Series(index=clin.index, dtype="float")
    y[(event == 1) & (months <= horizon_months)] = 1
    y[(event == 0) & (months >= horizon_months)] = 0
    y[(event == 1) & (months > horizon_months)] = 0
    keep = (er & y.notna()); y = y[keep].astype(int)
    feats = pd.concat([score_sets(Xg, SIGNATURES), score_sets(Xg, HALLMARK_SETS)], axis=1).loc[y.index]
    feats = feats.reindex(columns=list(feature_columns))
    return feats, y, Xg.loc[y.index]

d = load_checkpoint("lesson03_features")
X_genes, X_feat, clin_all, y_all = d["X_genes"], d["feats"], d["clin"], d["y"]
tr, va, te = d["tr"], d["va"], d["te"]

# survival frame (time-to-event) recovered from the clinical table, for the Part-4 survival cell
_status = next(c for c in ["RFS_STATUS", "DFS_STATUS"] if c in clin_all.columns)
_months = next(c for c in ["RFS_MONTHS", "DFS_MONTHS"] if c in clin_all.columns)
surv_all = pd.DataFrame({"months": pd.to_numeric(clin_all[_months], errors="coerce"),
                         "event":  clin_all[_status].astype(str).str.startswith("1").astype(int)}).loc[y_all.index]

# REAL external cohort (downloads GSE6532 SOFT ~180 MB on first run)
X_ext_feat, y_ext, X_ext_genes = load_gse6532_external(X_feat.columns, data_dir=DATA_DIR)
print(f"raw genes: {X_genes.shape} | engineered features: {X_feat.shape[1]}")
print(f"split: train {len(tr)} | val {len(va)} | test {len(te)} | recurrence {y_all.mean():.1%}")
print(f"external GSE6532 cohort (real, Affymetrix): {len(y_ext)} ER+ patients, recurrence {y_ext.mean():.1%}")


---
## Part 1 — Review the provided analysis  *(≈20 min)*

> **Submitted manuscript (abstract).** *"We report a novel 50-gene transcriptomic signature that predicts
> recurrence in HR+/HER2− early breast cancer with cross-validated AUC ≈ 0.69. After selecting the most
> recurrence-associated genes and normalising the cohort, a gradient-boosting classifier — with
> hyperparameters optimised by cross-validation — achieves what the authors call strong, clinically promising discrimination. The top genes are
> drivers of recurrence and represent promising therapeutic targets."*

The cell below is the authors' analysis, reproduced faithfully. **Run it and reproduce their headline
number.** Do not fix anything yet — your job in Part 2 is to find out *why* this number is not what it
seems.


In [ ]:
# ================= SUBMITTED ANALYSIS (as provided by the authors) =================
# Reproduce the headline number exactly as written. (We will audit this in Part 2.)
X_all = X_genes.copy()                       # raw genes, all patients
y = y_all.copy()

# "Normalise the cohort" — scaling fit on ALL samples at once
scaler_all = StandardScaler()
X_scaled = pd.DataFrame(scaler_all.fit_transform(X_all.fillna(X_all.median())),
                        index=X_all.index, columns=X_all.columns)

# "Select the most recurrence-associated genes" — supervised selection on ALL data
selector = SelectKBest(score_func=f_classif, k=50)
selector.fit(X_scaled, y)
top50 = X_scaled.columns[selector.get_support()]
X_sig = X_scaled[top50]

# "Gradient boosting, hyperparameters optimised by CV" — tuned and reported on the SAME CV
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
grid = {"n_estimators": [100, 300], "max_depth": [2, 3], "learning_rate": [0.05, 0.1]}
search = GridSearchCV(gb, grid, scoring="roc_auc",
                      cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
search.fit(X_sig, y)

headline_auc = search.best_score_
print(f"HEADLINE: cross-validated AUC = {headline_auc:.3f}  (50-gene signature, tuned gradient boosting)")
print(f"Reported 'driver' genes (top 8 of 50): {list(top50[:8])}")
# (Authors then describe these genes as 'drivers of recurrence and therapeutic targets'.)

> **Exercise 1.1 — restate the claim.** In one or two sentences, state the analysis's claim precisely:
> the prediction task, the cohort and *n*, the metric, the reported performance, and the biological
> assertion. (You'll test each of these in Part 2.)


**Claim (worked):** The authors claim a 50-gene transcriptomic signature predicts 5-year **recurrence**
in HR+/HER2− early breast cancer (METABRIC, n ≈ a few hundred) with a **cross-validated ROC-AUC ≈ 0.69**,
using a gradient-boosting classifier whose hyperparameters were optimised by cross-validation; and they
assert the top genes are **causal drivers** of recurrence and **therapeutic targets**. Performance is
reported on the discovery cohort only, with no external cohort and no clinical-variable baseline.


---
## Part 2 — Identify the methodological flaws  *(≈35 min — spine)*

Audit the submitted analysis against the **reviewer checklist**. The flaws are *not* labelled; you find
them. Then demonstrate *why* the leakage inflates the number.


> **Exercise 2.1 — the flaw table.** Read the Part 1 cell line by line and identify each methodological
> flaw: what it is, *where* it occurs, why it inflates / over-claims, and which lecture it violates. Fill
> in the table in the markdown cell below (aim for five).
>
> **Exercise 2.2 — demonstrate the leakage.** Show *why* selecting genes on all the data before CV
> inflates performance: re-run the authors' select-then-CV procedure on **permuted (shuffled) labels**.
> Honest pipelines score ≈ 0.5 on permuted labels; a leaky one scores well above 0.5.


In [ ]:
# 2.2 — permuted-label test of the authors' leaky procedure.
# If selecting features on ALL data (incl. the CV folds) leaks, it will "predict" even SHUFFLED labels.
def leaky_score(y_use):
    Xs = pd.DataFrame(StandardScaler().fit_transform(X_genes.fillna(X_genes.median())),
                      index=X_genes.index, columns=X_genes.columns)
    sel = SelectKBest(f_classif, k=50).fit(Xs, y_use)          # selection on ALL data (the leak)
    Xsig = Xs[Xs.columns[sel.get_support()]]
    return cross_val_score(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
                           Xsig, y_use, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                           scoring="roc_auc").mean()

real = leaky_score(y_all)
rng2 = np.random.default_rng(RANDOM_STATE)
perm = np.mean([leaky_score(pd.Series(rng2.permutation(y_all.values), index=y_all.index)) for _ in range(5)])
print(f"leaky procedure, REAL labels    : AUC {real:.3f}")
print(f"leaky procedure, PERMUTED labels: AUC {perm:.3f}   (an honest pipeline would be ~0.50)")
print(f"\nThe permuted-label AUC is well above 0.5 -> the select-on-all-data step LEAKS:")
print("the held-out folds influenced which genes were kept, so the CV estimate is optimistic.")

**Flaw table (worked):**

| # | Flaw | Where in the analysis | Why it inflates / over-claims | Violates |
|---|------|------------------------|-------------------------------|----------|
| F1 | **Feature-selection leakage** | `SelectKBest(...).fit(X_scaled, y)` on **all** data, before CV | the held-out folds helped choose the 50 genes, so the CV score is optimistic (see 2.2: high AUC even on permuted labels) | L3 |
| F2 | **No external validation** | only `cross_val_score` on METABRIC; GSE6532 never used | in-cohort CV says nothing about transfer; the claim of generalization is unsupported | L5 |
| F3 | **Incorrect preprocessing** | `StandardScaler().fit_transform(X_all)` on **all** data before any split | scaling statistics borrow from the held-out folds — another leak, and it would worsen across a real platform shift | L4 |
| F4 | **Overfitting / tuning optimism** | `GridSearchCV` tunes GBM and the **same** CV `best_score_` is reported; no nested CV | the reported number is the maximum over a tuning search — optimistic by selection | L4 |
| F5 | **Unsupported interpretation** | "top genes are drivers of recurrence and therapeutic targets" | importance/association is not causation, and no stability check is done; the genes may track grade/proliferation or be artefacts | L4 |


---
## Part 3 — Propose corrections  *(≈20 min)*

For each flaw, state the principled fix and the *predicted* effect on the reported performance.


> **Exercise 3.1 — corrections table.** Complete the table: flaw → fix → predicted effect (most fixes should *lower* the headline number).

**Corrections table (worked):**

| Flaw | Fix | Predicted effect |
|------|-----|------------------|
| F1 feature-selection leakage | move selection **inside** each CV fold (a `Pipeline` of select → model) | AUC drops toward the honest level |
| F2 no external validation | evaluate the locked model on **GSE6532** | a further, informative drop (the generalization gap) |
| F3 preprocessing leakage | fit the scaler **inside** the pipeline / on train only | small drop; matters more across the platform shift |
| F4 tuning optimism | report **nested CV** (or a one-time untouched test), not the tuned `best_score_` | the optimism gap is removed |
| F5 unsupported interpretation | report features as **predictive, not causal**; add a **stability** check | no performance change; the *claim* becomes defensible |


---
## Part 4 — Build a corrected workflow  *(≈45 min — spine)*

Implement the fixes: selection and scaling **inside** the CV; **nested CV** for tuning; an honest
baseline on the engineered features; a **stability** check on interpretation; and a genuine **external
validation** on GSE6532 plus a **clinical-baseline** comparison.


> **Exercise 4.1 — leakage-safe internal estimate.** Build a `Pipeline` that does scaling → supervised
> selection → model, so *everything refits inside each fold*. Report its honest cross-validated AUC on the
> raw genes; compare to the headline.
>
> **Exercise 4.2 — nested CV for the tuned model.** Wrap the tuned gradient-boosting search in nested CV
> and report the honest estimate (removes the Part-1 tuning optimism).
>
> **Exercise 4.3 — external validation + clinical baseline.** Fit an honest baseline (regularised LR) on
> the **engineered features** (train), and evaluate on the **external** GSE6532 cohort. Compare to a
> clinical-style baseline (here: the `proliferation` signature alone, a grade/Ki-67 proxy).
>
> **Exercise 4.4 (optional) — stability & survival.** Check how often the top selected genes recur across
> resamples; and (optional) fit a conceptual survival model on the time-to-event outcome.


> **Real-world exemplar — ER-Predict.** The course's published exemplar, the **ER-Predict** assay
> (Boscolo Bielo et al., *ESMO Open* 2026), was developed on this same METABRIC cohort and made exactly
> the research-grade choice the 4.4 cell points at. Its core is **time-dependent (survival)** — a
> **survival SVM** plus a **gradient-boosting survival forest** — and a time-independent **binary
> classifier is used only as a tiebreaker** when the two survival models disagree. So the binary recurrence
> label you have used all course is, in the real model, demoted to a supporting role behind a survival core.
>
> **Scope note (unchanged):** we do not teach survival mechanics (hazard functions, partial likelihood).
> The goal is to *recognise* when a binary endpoint throws away information and to *know* that time-to-event
> methods exist and were the right choice for the real model.


In [ ]:
CV = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# 4.1 — leakage-safe pipeline: scaling + selection INSIDE each fold
safe_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("select", SelectKBest(f_classif, k=50)),
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
safe_auc = cross_val_score(safe_pipe, X_genes, y_all, cv=CV, scoring="roc_auc").mean()
print(f"4.1 leakage-safe internal AUC (selection inside CV): {safe_auc:.3f}   vs headline {headline_auc:.3f}")

# 4.2 — nested CV for the tuned gradient-boosting model (honest, includes tuning cost)
inner = StratifiedKFold(4, shuffle=True, random_state=RANDOM_STATE)
gb_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                    ("select", SelectKBest(f_classif, k=50)),
                    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))])
gb_grid = {"clf__n_estimators": [100, 300], "clf__max_depth": [2, 3], "clf__learning_rate": [0.05, 0.1]}
nested = cross_val_score(GridSearchCV(gb_pipe, gb_grid, scoring="roc_auc", cv=inner),
                         X_genes, y_all, cv=CV, scoring="roc_auc")
print(f"4.2 nested-CV AUC (tuned GBM, honest)              : {nested.mean():.3f} ± {nested.std():.3f}")

# 4.3 — honest baseline on engineered features + EXTERNAL validation on GSE6532 + clinical baseline
def raw_lr():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("clf", LogisticRegression(C=0.5, max_iter=5000, random_state=RANDOM_STATE))])
base = raw_lr().fit(X_feat.loc[tr].fillna(X_feat.loc[tr].median()), y_all.loc[tr])
internal_feat = cross_val_score(raw_lr(), X_feat.loc[tr], y_all.loc[tr], cv=CV, scoring="roc_auc").mean()
ref_mu, ref_sd = X_feat.loc[tr].mean(), X_feat.loc[tr].std() + 1e-9
X_ext_corr = ((X_ext_feat - X_ext_feat.mean()) / (X_ext_feat.std() + 1e-9)) * ref_sd + ref_mu  # batch-correct on train ref
ext_auc = roc_auc_score(y_ext, base.predict_proba(X_ext_corr.fillna(0))[:, 1])
# clinical-style baseline: proliferation signature alone (a grade/Ki-67 proxy)
clin_base = raw_lr().fit(X_feat.loc[tr, ["proliferation"]], y_all.loc[tr])
clin_ext = roc_auc_score(y_ext, clin_base.predict_proba(X_ext_corr[["proliferation"]].fillna(0))[:, 1])
print(f"4.3 honest baseline internal AUC (engineered)     : {internal_feat:.3f}")
print(f"4.3 EXTERNAL AUC on GSE6532 (full model)          : {ext_auc:.3f}")
print(f"4.3 EXTERNAL AUC, clinical baseline (proliferation): {clin_ext:.3f}   <- must be beaten to add value")

# external validation of the AUTHORS' FLAWED raw-gene model -- does the leaky 50-gene signature transfer?
_mu = pd.Series(scaler_all.mean_, index=X_all.columns).reindex(top50)
_sd = pd.Series(scaler_all.scale_, index=X_all.columns).reindex(top50)
Xext_flawed = (X_ext_genes.reindex(columns=top50) - _mu) / _sd          # apply the authors' TRAIN-set scaling
flawed_ext_auc = roc_auc_score(y_ext, search.best_estimator_.predict_proba(Xext_flawed.fillna(0))[:, 1])
print(f"4.3 EXTERNAL AUC, AUTHORS' flawed 50-gene model   : {flawed_ext_auc:.3f}   <- raw-gene signature, cross-platform")

# 4.4 — stability of the selected genes across bootstraps
rngb = np.random.default_rng(RANDOM_STATE); top_counts = Counter()
Xg_imp = X_genes.fillna(X_genes.median())
for _ in range(20):
    bs = rngb.choice(X_genes.index, size=len(X_genes), replace=True)
    sel = SelectKBest(f_classif, k=50).fit(StandardScaler().fit_transform(Xg_imp.loc[bs]), y_all.loc[bs])
    for g in X_genes.columns[sel.get_support()]: top_counts[g] += 1
stable = pd.Series(top_counts).sort_values(ascending=False) / 20
print(f"\n4.4 selection stability: only {(stable >= 0.8).sum()} of 50 genes appear in >=80% of resamples "
      f"(top: {list(stable.head(3).round(2).items())})")

# 4.4 (optional) conceptual survival model
try:
    from lifelines import CoxPHFitter
    df = surv_all.loc[tr].join(X_feat.loc[tr, ["proliferation", "er_signalling"]]).dropna()
    cph = CoxPHFitter().fit(df, duration_col="months", event_col="event")
    print(f"4.4 (survival) proliferation hazard ratio = {np.exp(cph.params_['proliferation']):.2f} "
          f"(uses time-to-event + censoring the binary label discarded)")
except Exception as e:
    print(f"4.4 (survival) lifelines not available — skipping the optional Cox model ({type(e).__name__}).")

---
## Part 5 — Compare original vs corrected  *(≈25 min)*

Put the inflated original number next to the honest corrected numbers, and quantify how much was real
signal versus methodological artefact.


> **Exercise 5.1 — comparison.** Build a small table/figure: original (leaky) AUC vs corrected internal
> (leakage-safe + nested) vs corrected **external** AUC. Quantify the inflation (original − honest).
>
> **Exercise 5.2 — interpretation.** In one paragraph: how much of the original “performance” was real
> signal, and how much was leakage + tuning optimism + the absence of external validation?


In [ ]:
rows = [
    ("original (leaky, tuned, internal)", headline_auc),
    ("flawed model EXTERNAL (raw genes, GSE6532)", flawed_ext_auc),
    ("corrected internal — leakage-safe", safe_auc),
    ("corrected internal — nested CV (tuned)", nested.mean()),
    ("corrected EXTERNAL (engineered, GSE6532)", ext_auc),
    ("clinical baseline (external)", clin_ext),
]
tbl = pd.DataFrame(rows, columns=["analysis", "AUC"])
print(tbl.round(3).to_string(index=False))
print(f"\nThe authors' raw-gene model collapses externally ({headline_auc:.3f} -> {flawed_ext_auc:.3f}):")
print("a leaky signature of platform-specific probes does not transfer across cohorts. The corrected")
print(f"engineered-feature model HOLDS externally ({ext_auc:.3f}) -- robust biological features cross")
print(f"platforms (Lecture 3) -- but barely beats the clinical baseline ({clin_ext:.3f}), so its *incremental*")
print("value over standard clinical variables is the real open question.")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
colors = ["#A8403A", "#C75D49", "#0E7C86", "#0E7C86", "#2E7D57", "#9aa7ab"]
ax.barh([r[0] for r in rows][::-1], [r[1] for r in rows][::-1], color=colors[::-1])
ax.axvline(0.5, color="#5B6B70", ls="--", lw=1)
ax.set_xlim(0.4, 1.0); ax.set_xlabel("ROC-AUC"); ax.set_title("Original (leaky) vs corrected (honest)")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

# 5.2 interpretation (example): the leaky headline shrinks once selection/scaling move inside CV (F1/F3)
# and tuning is estimated by nested CV (F4). The decisive test is EXTERNAL: the authors' raw-gene
# signature collapses on GSE6532 (F2) -- platform-specific probes do not transfer -- while the corrected
# engineered-feature model holds, because biological signatures are cross-platform robust (Lecture 3).
# The remaining question is incremental value: it barely beats a proliferation/clinical baseline.

---
## Part 6 — Write a reviewer report  *(≈30 min — the primary deliverable)*

Using the reviewer checklist as a template, write a structured review of the **original** submission.
Sections: summary of the claim · major issues (with evidence) · minor issues · reproducibility assessment
· clinical-relevance assessment · **recommendation** (accept / minor revision / major revision / reject)
with justification. End with: *Would you recommend publication? Why or why not?*


**Reviewer report (worked example):**

**Summary of claim.** The authors propose a 50-gene signature predicting 5-year recurrence in HR+/HER2−
breast cancer (METABRIC), reporting a cross-validated ROC-AUC ≈ 0.69 with a tuned gradient-boosting model,
and describe the top genes as causal drivers and therapeutic targets.

**Major issues.**
1. *Feature-selection leakage (F1).* Genes are selected on the entire dataset before cross-validation, so
   the reported CV score is optimistic. A permuted-label test of the authors' own procedure yields an AUC
   well above 0.5 (Part 2.2), confirming the leak. When selection is moved inside each fold, the internal
   AUC falls only modestly (≈0.69 → 0.66) — the leak is real but explains only part of the headline.
2. *No external validation (F2).* Performance is reported only on the discovery cohort. On an independent
   (GSE6532-style, different-platform) cohort the authors' raw-gene model collapses to AUC ≈ 0.47 (no better than chance and below the clinical baseline) — the generalization gap the
   manuscript never reports. (A corrected engineered-feature model does transfer at ≈0.69, but only matches a proliferation-only clinical baseline at ≈0.68.)
3. *Preprocessing leakage (F3).* Normalisation is fit on all samples at once; it must be fit on training
   data only and applied to held-out/external data.
4. *Tuning optimism (F4).* Hyperparameters were optimised on the same cross-validation that is reported;
   the honest, nested-CV estimate is lower.
5. *Unsupported interpretation (F5).* "Drivers of recurrence / therapeutic targets" is a causal claim from
   a predictive model, with no stability check. In fact only about half of the 50 genes (25/50) recur reliably across resamples
   (Part 4.4); the selection is only partly stable and the genes plausibly track grade/proliferation.

**Minor issues.** No comparison to standard clinical variables; no effect-size / clinical-utility analysis;
the binary 5-year label discards late recurrences (a survival framing would be more appropriate).

**Reproducibility.** Seeds/environment are not reported and preprocessing is entangled with the split;
the analysis is not, as written, reliably regenerable.

**Clinical relevance.** Not established: the model is not shown to beat a simple clinical baseline
(here the proliferation signature) on the external cohort, and no decision-changing effect is demonstrated.

**Recommendation: Reject (major flaws), with an invitation to resubmit.** The central claim is not
supported by the evidence: the headline AUC is modest (~0.69) and only partly inflated by leakage and tuning optimism (it eases under
a leakage-safe, nested analysis to ≈0.66); more decisively, the raw-gene signature fails to transfer externally (≈0.47) and the corrected model does not beat a clinical baseline. The
causal language is unsupported. A resubmission would need selection/scaling/correction inside the
validation loop, nested CV or an untouched test set, genuine external validation, a clinical-baseline
comparison, predictive (not causal) interpretation with stability, and — given late recurrence in HR+
disease — ideally a survival framing. **I would not recommend publication in the current form.**


---
## Part 7 — Reflection  *(≈15 min)*

A short written reflection (commit it to the notebook): which course lessons (L1–L5) were most important in
catching these flaws, and which do you expect to use most in your own research?


**Reflection (worked example).** The decisive lessons were L3's feature-selection leakage (the headline
propped up part of the internal headline by selecting genes outside the validation loop (honest internal AUC ≈0.66, not the reported ≈0.69)), L4's nested CV and correct
preprocessing placement (tuning optimism and scaling leakage), and L5's external validation and
significance-vs-utility framing (the model neither transferred nor beat a clinical baseline). L4's
prediction-vs-causation point exposed the over-claimed "drivers." In my own work the habits I'll carry are
encapsulating *everything* data-dependent in a pipeline that refits inside CV, treating an external cohort
as untouchable until the very end, and refusing to write a causal sentence from a model weight.

> **Discussion.** If this study were resubmitted with all corrections, the most it could honestly claim is
> a modest, externally-validated prognostic signal that should be compared to — and shown to add value
> over — standard clinical variables, and framed as predictive, not causal. Whether that is publishable
> depends on whether it beats the clinical baseline and changes a decision.


---
### Deliverables checklist
- [ ] Reproduced headline number + restated claim (Part 1)
- [ ] Flaw table (5 flaws) + permuted-label leakage demonstration (Part 2)
- [ ] Corrections table with predicted effects (Part 3)
- [ ] Corrected workflow: leakage-safe internal, nested CV, external (GSE6532), clinical baseline, stability (Part 4)
- [ ] Original-vs-corrected comparison + inflation quantified (Part 5)
- [ ] Structured reviewer report with a justified recommendation (Part 6)
- [ ] Reflection on the course's key lessons (Part 7)

> **The message, in one line:** *the hardest part of ML is not training a model; it is demonstrating that
> the model is trustworthy.* You built the corrected workflow — and the reviewer report is where you prove
> you can also detect the absence of trust. That is the whole course, exercised once.
